In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
import joblib


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Sto addestrando su: {device}")

Sto addestrando su: mps


In [ ]:
#@title Caricamento e preparazione dati
df = pd.read_csv('AI_machine_learning_data_set.csv')
    
# Tolgo le colonne che non servono al modello (identificativi e flag non predittivi)
df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud', 'step'])

# Separo le feature (X) dalla variabile target (y)
X = df_model.drop('isFraud', axis=1)

y = df_model['isFraud'] 

# Divido in train e test (30% test), mantenendo stratificata la percentuale di frodi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


# UNDER-SAMPLING (BILANCIAMENTO DEL TRAINING SET)
# Il dataset è molto sbilanciato: bilancio solo il training set, lasciando intatto il test

# Riunisco momentaneamente X_train e y_train per poterli filtrare insieme
df_train_completo = pd.concat([X_train, y_train], axis=1)

# Separo le frodi dalle transazioni lecite
frodi_train = df_train_completo[df_train_completo['isFraud'] == 1]
non_frodi_train = df_train_completo[df_train_completo['isFraud'] == 0]

# Under-sampling: prendo a caso tante transazioni lecite quante sono le frodi
non_frodi_bilanciate = non_frodi_train.sample(n=len(frodi_train), random_state=42)

# Unisco le due parti nel nuovo training set bilanciato e mescolo le righe
df_train_bilanciato = pd.concat([frodi_train, non_frodi_bilanciate]).sample(frac=1, random_state=42)

# Ricavo X e y bilanciate, che userò per l'addestramento della rete neurale
x_train_bilanciato = df_train_bilanciato.drop('isFraud', axis=1)
y_train_bilanciato = df_train_bilanciato['isFraud']

print("--- CONTEGGIO BILANCIATO TRAINING SET (DEEP LEARNING) ---")
print(y_train_bilanciato.value_counts())


# PREPROCESSING
# Standardizzo le colonne numeriche e codifico quella categorica in variabili dummy

categorical_cols = ['type']
numeric_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)


# Applico il preprocessor: lo "alleno" sul training set bilanciato...
X_train_processed = preprocessor.fit_transform(x_train_bilanciato)

# ...e lo applico al test set senza modificarlo, per simulare dati mai visti (come nel mondo reale)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
#@title Conversione in tensori PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader

# CONVERSIONE IN MATRICI DENSE
# L'OneHotEncoder può restituire una matrice sparsa: la convertiamo in densa con .toarray(),
# perché PyTorch lavora con array "normali"
if hasattr(X_train_processed, "toarray"):
    X_train_dense = X_train_processed.toarray()
else:
    X_train_dense = X_train_processed

if hasattr(X_test_processed, "toarray"):
    X_test_dense = X_test_processed.toarray()
else:
    X_test_dense = X_test_processed


# CREAZIONE DEI TENSORI PYTORCH

# Tensori di input e target per il TRAINING (uso i dati bilanciati)
X_train_tensor = torch.tensor(X_train_dense, dtype=torch.float32)
# .values estrae l'array numpy puro da y_train_bilanciato, necessario per costruire il tensore
y_train_tensor = torch.tensor(y_train_bilanciato.values, dtype=torch.float32).unsqueeze(1)

# Tensori di input e target per il TEST (uso i dati originali, non bilanciati)
X_test_tensor = torch.tensor(X_test_dense, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


# CREAZIONE DEL DATALOADER
# Il DataLoader passa i dati al modello a piccoli gruppi (batch) durante l'addestramento

batch_size = 2048
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print("Tensori e DataLoader creati con successo!")
print(f"Dimensioni X_train_tensor: {X_train_tensor.shape}")
print(f"Dimensioni y_train_tensor: {y_train_tensor.shape}")

In [ ]:
#@title Definizione della rete neurale
class FraudDetectionNN(nn.Module):
    def __init__(self, input_dim):
        super(FraudDetectionNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout più alto nei primi strati, per limitare l'overfitting
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.1), # Dropout più leggero verso l'uscita
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_tensor.shape[1]
model = FraudDetectionNN(input_dim)
model.to(device)

In [ ]:
#@title Configurazione addestramento
# Loss function per classificazione binaria (frode/lecita), senza pesi aggiuntivi:
# lo sbilanciamento è già gestito a monte con l'under-sampling
criterion = nn.BCEWithLogitsLoss()

# Optimizer Adam, con un learning rate basso per un addestramento più stabile
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Configurazione completata: Loss Function senza pesi artificiali.")

In [ ]:
#@title Training loop
epochs = 100

print("Inizio Addestramento...")
for epoch in range(epochs):
    model.train()  # Modalità training: attiva Dropout e BatchNorm
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()  # Azzero i gradienti del batch precedente
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()  # Calcolo i gradienti
        optimizer.step()  # Aggiorno i pesi del modello
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoca [{epoch+1}/{epochs}], Loss Media: {avg_loss:.4f}")

In [ ]:
#@title Valutazione del modello
model.eval()  # Modalità valutazione: disattiva Dropout e BatchNorm

X_test_device = X_test_tensor.to(device)
y_test_device = y_test_tensor.to(device)

with torch.no_grad():
    # 1. Calcolo le predizioni sui dati di test
    test_outputs = model(X_test_device)
    
    # 2. Applico la sigmoide per trasformare l'output in una probabilità tra 0 e 1
    probabilities = torch.sigmoid(test_outputs)
    
    # 3. Classifico come frode solo se la probabilità supera 0.8 (soglia più alta del solito 0.5,
    #    per essere più selettivi e ridurre i falsi positivi)
    y_pred_tensor = (probabilities >= 0.8).float()
    
    # 4. Calcolo l'accuratezza finale sul test set
    corrette = (y_pred_tensor == y_test_device).sum().item()
    totali = y_test_device.size(0)
    accuratezza_finale = corrette / totali
    print(f"\nACCURATEZZA GLOBALE SUL TEST SET: {accuratezza_finale:.2%}")
    
    # Per usare scikit-learn (matrice di confusione/report) servono array numpy 1D su CPU
    y_pred = y_pred_tensor.cpu().numpy().astype(int).ravel()

print(f"\nAccuratezza Finale sul Test Set: {accuratezza_finale * 100:.2f}%")

# Matrice di confusione
cm = confusion_matrix(y_test, y_pred)

# La trasformo in un DataFrame con etichette leggibili per righe e colonne
cm_con_titoli = pd.DataFrame(cm, 
                             columns=['PREDETTO LECITO', 'PREDETTO FRODE'], 
                             index=['REALE LECITO', 'REALE FRODE'])

# Stampo a schermo la matrice formattata
print("\nMATRICE DI CONFUSIONE:")
print(cm_con_titoli)

print("\nReport di Classificazione:")
print(classification_report(y_test, y_pred))



In [ ]:
#@title Salvataggio modello e preprocessor
import joblib
import torch

# 1. Salvo il preprocessor (contiene lo StandardScaler e l'OneHotEncoder già addestrati)
joblib.dump(preprocessor, 'preprocessor.pkl')

# 2. Salvo i pesi della rete neurale addestrata
torch.save(model.state_dict(), 'fraud_detection_pytorch.pth')

print("✅ File 'preprocessor.pkl' e 'fraud_detection_pytorch.pth' creati e salvati con successo!")